# AOGEC + Error Type Classifier using T5

In [ ]:
# 0) Install dependencies
!pip -q install evaluate seaborn sacrebleu errant
!python -m spacy download en_core_web_sm -q

In [ ]:
# 1) Import Libraries
import os, re, json, random, collections, math
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional, Any
from tqdm.auto import tqdm
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
 
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score, confusion_matrix,
)
import evaluate
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

In [31]:
# 2) Reproducibility
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# 3) Configuration
@dataclass
class CFG:
    # ── Model ──────────────────────────────────────────────────────
    model_name: str = "t5-base"
    device: torch.device = field(
        default_factory=lambda: torch.device("cuda" if torch.cuda.is_available() else "cpu")
    )
 
    # ── Data ───────────────────────────────────────────────────────
    data_dir: str = "/kaggle/input/datasets/solanadh/ao-gec-dataset-with-error-type"

    # ── Sequence lengths ───────────────────────────────────────────
    max_input_len:  int = 128
    max_target_len: int = 160
 
    # ── Optimisation ──────────────────────────────────────────────
    batch_size:       int   = 8
    grad_accum_steps: int   = 4          
    
    lr_encoder:       float = 2e-5
    lr_decoder:       float = 3e-5
    lr_cls_head:      float = 2e-4
    
    weight_decay:     float = 0.01
    warmup_ratio:     float = 0.06 
    grad_clip:        float = 1.0
    num_workers:      int   = 2
    label_smoothing:  float = 0.1
 
    # ── Mixed precision (OPT-1) ────────────────────────────────────
    # "bf16" on A100/H100, "fp16" on T4/V100, "none" on CPU
    amp_dtype: str = "bf16"
 
    # ── Gradient checkpointing (OPT-4) ────────────────────────────
    grad_ckpt: bool = True
 
    # ── Epochs ─────────────────────────────────────────────────────
    epochs_stage0: int = 4   # cls warm-up (decoder frozen here)
    epochs_stage1: int = 10   # joint training
    epochs_stage2: int = 3   # generation refinement
 
    # ── Early stopping ─────────────────────────────────────────────
    patience: int = 5
 
    # ── Loss weights ──────────────────────────────────────────────
    alpha_gen:       float = 2.0
    beta_cls_stage0: float = 1.5
    beta_cls_stage1: float = 0.4
    focal_gamma:     float = 0.2
 
    # ── Generation ────────────────────────────────────────────────
    num_beams_train:    int   = 4      
    num_beams_test:     int   = 4     
    min_length:         int   = 5
    repetition_penalty: float = 1.2
    no_repeat_ngram:    int   = 3
    length_penalty:     float = 0.8
 
    # ── GLEU ──────────────────────────────────────────────────────
    gleu_n: int = 4
 
    # ── Output ────────────────────────────────────────────────────
    output_dir: str = "/kaggle/working/aogec_output"
    best_ckpt:  str = ""
 
    def __post_init__(self):
        self.train_path = f"{self.data_dir}/AOGEC_Dataset_Labeled_train.csv"
        self.val_path   = f"{self.data_dir}/AOGEC_Dataset_Labeled_eval.csv"
        self.test_path  = f"{self.data_dir}/AOGEC_Dataset_Labeled_test.csv"
        self.best_ckpt  = os.path.join(self.output_dir, "best_model.pt")
        os.makedirs(self.output_dir, exist_ok=True)
        print(f"Device : {self.device}")
        if self.device.type == "cuda":
            print(f"GPU    : {torch.cuda.get_device_name(0)}")            
            if not torch.cuda.is_bf16_supported():
                self.amp_dtype = "fp16"
                print("GPU does not support bf16 → switched to fp16")
        else:
            self.amp_dtype = "none"
            self.grad_ckpt = False
            print("CPU mode — AMP and grad-ckpt disabled")
        print(f"AMP    : {self.amp_dtype}")
 
cfg = CFG()
 
# AMP context and scaler
_AMP_ENABLED = cfg.amp_dtype != "none"
_AMP_DTYPE   = torch.bfloat16 if cfg.amp_dtype == "bf16" else torch.float16
_autocast    = lambda: torch.amp.autocast(device_type="cuda", dtype=_AMP_DTYPE, enabled=_AMP_ENABLED)
scaler       = GradScaler(enabled=(_AMP_ENABLED and cfg.amp_dtype == "fp16"))

In [ ]:
# 4) GLEU Implementation
class GLEUScorer: 
    def __init__(self, n: int = 4):
        self.n = n
 
    @staticmethod
    def _tok(text: str) -> List[str]:
        return text.strip().split()
 
    @staticmethod
    def _ngrams(tokens: List[str], n: int) -> collections.Counter:
        return collections.Counter(
            tuple(tokens[i: i + n])
            for i in range(max(0, len(tokens) - n + 1))
        )
 
    def _sent_gleu(self, src: List[str], hyp: List[str], refs: List[List[str]]) -> float:
        if not hyp:
            return 0.0
        best = 0.0
        for ref in refs:
            total_match = total_possible = 0
            for k in range(1, self.n + 1):
                hyp_ng = self._ngrams(hyp, k)
                ref_ng = self._ngrams(ref, k)
                total_match    += sum(min(c, ref_ng[g]) for g, c in hyp_ng.items())
                total_possible += max(len(hyp) - k + 1, len(ref) - k + 1, 1)
            score = (total_match / total_possible) if total_possible else 0.0
            if hyp == src and hyp != ref:
                score = 0.0
            best = max(best, score)
        return best
 
    def corpus_gleu(self, sources: List[str], hypotheses: List[str],
                    references: List[List[str]]) -> float:
        assert len(sources) == len(hypotheses) == len(references)
        if not hypotheses:
            return 0.0
        total = sum(
            self._sent_gleu(self._tok(s), self._tok(h), [self._tok(r) for r in rs])
            for s, h, rs in zip(sources, hypotheses, references)
        )
        return round(total / len(hypotheses) * 100.0, 4)
 
    def sentence_gleu(self, source: str, hypothesis: str, references: List[str]) -> float:
        return self._sent_gleu(
            self._tok(source), self._tok(hypothesis),
            [self._tok(r) for r in references],
        ) * 100.0

In [ ]:
# 5) Initialize scorers
gleu_scorer = GLEUScorer(n=cfg.gleu_n)
chrf_metric = evaluate.load("chrf")
print("GLEU scorer and chrF metric ready.")


In [ ]:
# 6) Data Preprocessing & Label Encoding
# ------------------------------
# Text Normalization
# ------------------------------
def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s([?.!,;:])", r"\1", text)
    return text.strip()
 
def clean_df(df: pd.DataFrame) -> pd.DataFrame:
    required = ["input", "target", "error_type"]
    df = df[required].copy()
    df.dropna(subset=required, inplace=True)
    for col in ["input", "target"]:
        df[col] = df[col].astype(str).apply(normalize_text)
    df["error_type"] = df["error_type"].astype(str).str.strip().str.lower()
    
    df.drop_duplicates(subset=required, inplace=True)
    return df.reset_index(drop=True)
 
 
train_df = clean_df(pd.read_csv(cfg.train_path, encoding="utf-8", encoding_errors="ignore"))
val_df   = clean_df(pd.read_csv(cfg.val_path,   encoding="utf-8", encoding_errors="ignore"))
test_df  = clean_df(pd.read_csv(cfg.test_path,  encoding="utf-8", encoding_errors="ignore"))

labels_sorted_train = sorted(train_df["error_type"].unique().tolist())
label2id = {lbl: i for i, lbl in enumerate(labels_sorted_train)}
id2label = {i: lbl for lbl, i in label2id.items()} 
 
def map_labels(df: pd.DataFrame, mapping: dict, default: str, name: str = "") -> pd.DataFrame:
    unseen = set(df["error_type"]) - set(mapping)
    if unseen:
        print(f"{name}: unseen labels {unseen}")
    df = df.copy()
    df["error_id"] = df["error_type"].apply(lambda x: mapping.get(x, mapping.get(default, 0)))
    return df
 
 
default_label = labels_sorted_train[0]
train_df = map_labels(train_df, label2id, default_label, "train")
val_df   = map_labels(val_df,   label2id, default_label, "val")
test_df  = map_labels(test_df,  label2id, default_label, "test")
 
print(f"Data Desc. | Train={len(train_df)}  Val={len(val_df)}  Test={len(test_df)}")
print(f"Labels ({len(labels_sorted_train)}): {labels_sorted_train}")
print("\nTrain Dataset class distribution:")
print(train_df["error_type"].value_counts().to_string())

In [ ]:
# 7) Tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
_PAD_ID   = tokenizer.pad_token_id

In [ ]:
# 8) Helpers
def build_neutral_prompt(text: str) -> str:
    """
    Classification pass — no error_type in the token sequence.
    The model must infer error type purely from linguistic content.
    Used at BOTH train-time and inference-time for the cls head.
    """
    return f"classify the grammatical error in afan oromo: {text}"

def build_labeled_prompt(text: str, error_type: str) -> str:
    """
    Generation pass — includes the error type label.
    During training the ground-truth label is used.
    During inference the predicted label (from cls head) is used.
    Decoder is always trained on this distribution → no train/test mismatch.
    """
    # if error_type == "no-error":
    #     return f"do not change this sentence in afan oromo: {text}"
    return f"correct {error_type} in afan oromo: {text}"

def post_process(text: str) -> str:
    text = re.sub(r'\s([?.!,;:](?:\s|$))', r'\1', text)
    return text.strip()

In [ ]:
# 9. Dataset
# ------------------------------
class DualPromptGECDataset(Dataset):
    """
    Stores two pre-tokenised views per sample:
      1. neutral_ids  — neutral prompt → cls encoder stream
      2. labeled_ids  — label-conditioned prompt → gen decoder stream
    """
 
    def __init__(self, df: pd.DataFrame, tokenizer, max_in: int, max_out: int):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_in    = max_in
        self.max_out   = max_out
        self._build()
 
    def _encode(self, text: str, max_len: int) -> Dict[str, torch.Tensor]:
        enc = self.tokenizer(
            text, max_length=max_len, truncation=True,
            padding="max_length", return_tensors="pt",
        )
        return {k: v.squeeze(0) for k, v in enc.items()}
 
    def _build(self):
        self.neutral_ids    = []
        self.neutral_masks  = []
        self.labeled_ids    = []
        self.labeled_masks  = []
        self.decoder_labels = []
        self.error_labels   = []
 
        for i in tqdm(range(len(self.df)), desc="Tokenising", leave=False):
            row       = self.df.iloc[i]
            text      = str(row["input"])
            target    = str(row["target"])
            error_id  = int(row["error_id"])
            error_str = str(row["error_type"])
 
            n_enc = self._encode(build_neutral_prompt(text), self.max_in)
            l_enc = self._encode(build_labeled_prompt(text, error_str), self.max_in)
            t_enc = self._encode(target, self.max_out)
 
            lbl = t_enc["input_ids"].clone()
            lbl[lbl == _PAD_ID] = -100
 
            self.neutral_ids.append(n_enc["input_ids"])
            self.neutral_masks.append(n_enc["attention_mask"])
            self.labeled_ids.append(l_enc["input_ids"])
            self.labeled_masks.append(l_enc["attention_mask"])
            self.decoder_labels.append(lbl)
            self.error_labels.append(torch.tensor(error_id, dtype=torch.long))
 
    def __len__(self) -> int:
        return len(self.df)
 
    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        return {
            "neutral_ids":  self.neutral_ids[idx],
            "neutral_mask": self.neutral_masks[idx],
            "labeled_ids":  self.labeled_ids[idx],
            "labeled_mask": self.labeled_masks[idx],
            "labels":       self.decoder_labels[idx],
            "error_labels": self.error_labels[idx],
            "raw_input":    row["input"],
            "raw_target":   row["target"],
            "raw_error":    row["error_type"],
        }
 
train_ds = DualPromptGECDataset(train_df, tokenizer, cfg.max_input_len, cfg.max_target_len)
val_ds   = DualPromptGECDataset(val_df,   tokenizer, cfg.max_input_len, cfg.max_target_len)
test_ds  = DualPromptGECDataset(test_df,  tokenizer, cfg.max_input_len, cfg.max_target_len)
print("Datasets ready.")

In [ ]:
# 10) DataLoaders
_loader_kwargs = dict(
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    pin_memory=(cfg.device.type == "cuda"),
    persistent_workers=(cfg.num_workers > 0),   # OPT-2
    prefetch_factor=(2 if cfg.num_workers > 0 else None),
)
train_loader = DataLoader(train_ds, shuffle=True,  drop_last=True,  **_loader_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=False, **_loader_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **_loader_kwargs)
print(f"DataLoaders ready | {len(train_loader)} train batches")

In [ ]:
# 11) Model 
class T5MultiTask(nn.Module):
    """
    Dual-stream T5 multi-task model for GEC + error type classification.
 
    Architecture

    Shared T5 encoder weights feed two streams:
      Stream A: neutral_prompt → encoder → mean+max pool → MLP → cls logits
      Stream B: labeled_prompt → encoder → T5 decoder   → correction tokens
    """
 
    def __init__(self, model_name: str, num_labels: int,
                 dropout: float = 0.2, label_smoothing: float = 0.1,
                 grad_ckpt: bool = False):
        super().__init__()
        self.seq2seq         = T5ForConditionalGeneration.from_pretrained(model_name)
        self.label_smoothing = label_smoothing
        H = self.seq2seq.config.d_model
 
        if grad_ckpt:
            self.seq2seq.encoder.gradient_checkpointing_enable()
            print("Gradient checkpointing enabled on encoder.")
 
        self.dropout = nn.Dropout(dropout)
 
        self.classifier = nn.Sequential(
            nn.Linear(2 * H, H),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(H),
            
            nn.Linear(H, H // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            
            nn.Linear(H // 2, H // 4),
            nn.GELU(),
            nn.Dropout(dropout),
            
            nn.Linear(H // 4, num_labels),
        )
 
    @staticmethod
    def _pool(hidden: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        """Mean + max pooling over unmasked positions."""
        m      = mask.unsqueeze(-1).type_as(hidden)
        mean_p = (hidden * m).sum(1) / m.sum(1).clamp(min=1e-9)
        max_p  = hidden.masked_fill(m == 0, -1e9).max(1).values
        return torch.cat([mean_p, max_p], dim=-1)
 
    def encode(self, input_ids: torch.Tensor, attention_mask: torch.Tensor):
        return self.seq2seq.encoder(
            input_ids=input_ids, attention_mask=attention_mask, return_dict=True
        )
 
    def forward(
        self,
        neutral_ids:  torch.Tensor,
        neutral_mask: torch.Tensor,
        labeled_ids:  torch.Tensor,
        labeled_mask: torch.Tensor,
        labels:       Optional[torch.Tensor] = None,
    ) -> Dict[str, Optional[torch.Tensor]]:
        # Stream A — neutral prompt → classify
        neutral_enc = self.encode(neutral_ids, neutral_mask)
        enc_pooled  = self._pool(neutral_enc.last_hidden_state, neutral_mask)
        # enc_pooled  = self._pool(neutral_enc.last_hidden_state.detach(), neutral_mask) 
        cls_logits  = self.classifier(self.dropout(enc_pooled))
        
        # Stream B — labeled prompt → generate; reuse encoder output
        labeled_enc = self.encode(labeled_ids, labeled_mask)
        seq_out = self.seq2seq(
            input_ids=labeled_ids,
            attention_mask=labeled_mask,
            encoder_outputs=labeled_enc,
            labels=labels,
            return_dict=True,
        )
 
        return {
            "gen_loss":   seq_out.loss if labels is not None else None,
            "cls_logits": cls_logits,
        }
 
    # ── Freeze / unfreeze helpers ──────────────────────────────────
    def freeze_decoder(self):
        for name, p in self.named_parameters():
            if "decoder" in name or "lm_head" in name:
                p.requires_grad_(False)
 
    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad_(True)
 
    # ── Batched two-pass inference ────────────────────────
    @torch.no_grad()
    def generate_and_classify(
        self,
        neutral_ids:  torch.Tensor,
        neutral_mask: torch.Tensor,
        raw_inputs:   List[str],
        tokenizer,
        cfg,
        id2label:     Dict[int, str],
        num_beams:    int = 1,
    ) -> Tuple[torch.Tensor, List[str]]:

        gen_kwargs = dict(
            max_new_tokens=cfg.max_target_len,
            min_length=cfg.min_length,
            num_beams=num_beams,
            repetition_penalty=cfg.repetition_penalty,
            no_repeat_ngram_size=cfg.no_repeat_ngram,
            length_penalty=cfg.length_penalty,
            early_stopping=(num_beams > 1),
        )
 
        # ── Pass 1: classify (one encoder call for the whole batch) ──
        neutral_enc = self.encode(neutral_ids, neutral_mask)
        enc_pooled  = self._pool(neutral_enc.last_hidden_state, neutral_mask)
        cls_preds   = torch.argmax(self.classifier(enc_pooled), dim=-1)
 
        # ── Pass 2: build labeled prompts from predicted labels ──────
        labeled_prompts = [
            build_labeled_prompt(raw, id2label[cls_preds[i].item()])
            for i, raw in enumerate(raw_inputs)
        ]
 
        # Tokenise the full batch at once
        batch_enc = tokenizer(
            labeled_prompts,
            return_tensors="pt",
            padding=True,  
            truncation=True,
            max_length=cfg.max_input_len,
        ).to(neutral_ids.device)
 
        out_ids = self.seq2seq.generate(
            input_ids=batch_enc["input_ids"],
            attention_mask=batch_enc["attention_mask"],
            **gen_kwargs,
        )
 
        gen_texts = [
            post_process(tokenizer.decode(ids, skip_special_tokens=True))
            for ids in out_ids
        ]
        return cls_preds, gen_texts

In [ ]:
# 12) Focal Loss
class FocalLoss(nn.Module):
    def __init__(self, gamma: float = 2.0, weight: Optional[torch.Tensor] = None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight
 
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # Upcast to float32 for numerical stability
        logits  = logits.float()
        targets = targets.view(-1)
 
        log_probs = F.log_softmax(logits, dim=-1)
        probs     = torch.exp(log_probs)
 
        log_pt = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt     = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
 
        focal = -((1 - pt) ** self.gamma) * log_pt
        if self.weight is not None:
            focal = focal * self.weight.gather(0, targets)
        return focal.mean()

In [ ]:
# 13) Model + loss initialisation
num_labels    = len(label2id)
labels_sorted = [id2label[i] for i in range(num_labels)]
 
model = T5MultiTask(
    cfg.model_name,
    num_labels=num_labels,
    label_smoothing=cfg.label_smoothing,
    grad_ckpt=cfg.grad_ckpt,
).to(cfg.device)
 
# Class-balanced focal loss weights
counts = (
    train_df.groupby("error_id").size()
    .reindex(range(num_labels), fill_value=0).values
)
cls_weights = torch.tensor(
    counts.max() / (counts + 1e-6), dtype=torch.float, device=cfg.device
)
cls_weights = cls_weights / cls_weights.mean()
 
focal_loss_fn = FocalLoss(gamma=cfg.focal_gamma, weight=cls_weights)
print(f"Model ready | params={sum(p.numel() for p in model.parameters()):,}")
print(f"Class weights: {dict(zip(labels_sorted, cls_weights.cpu().numpy().round(3)))}")

In [ ]:
# 14) Optimizer + Scheduler
def build_optimizer_and_scheduler(
    model, n_batches: int, n_epochs: int, stage: int = 1
):
    """
    Dynamically freezes/unfreezes layers according to the training stage
    to ensure the model learns classification first before joint multi-task generation.
    """
    enc_p, dec_p, cls_p = [], [], []
    seen = set()
 
    def _add(p, bucket):
        if id(p) not in seen:
            seen.add(id(p))
            bucket.append(p)
 
    # 1. Separate model parameters into explicit logical functional blocks
    for name, param in model.named_parameters():
        if "classifier" in name:
            _add(param, cls_p)
        elif "decoder" in name or "lm_head" in name:
            _add(param, dec_p)
        else:
            _add(param, enc_p)

    # 2. Configure architectural behavior per stage
    # Stage 0: CLASSIFICATION Starting (Freeze Encoder & Decoder entirely, optimize ONLY the classifier head)
    # Stage 1: JOINT MULTI-TASK TRAINING (Unfreeze everything to learn both tasks simultaneously)
    # Stage 2: GENERATION REFINEMENT (Fine-tune text generation, lowering encoder/classifier updates)
    if stage == 0:
        for p in enc_p: p.requires_grad = True
        for p in dec_p: p.requires_grad = False
        for p in cls_p: p.requires_grad = True
        
        lr_scales = (1.0, 0.0, 1.0) 
        
    elif stage == 1:
        for p in enc_p: p.requires_grad = True
        for p in dec_p: p.requires_grad = True
        for p in cls_p: p.requires_grad = True
        
        lr_scales = (1.0, 1.0, 1.0)
        
    elif stage == 2:
        for p in enc_p: p.requires_grad = True
        for p in dec_p: p.requires_grad = True
        for p in cls_p: p.requires_grad = True
        
        lr_scales = (0.5, 1.0, 0.1)

    # 3. Construct specific parameter group profiles (Filter out non-grad params to save optimizer state memory)
    groups = []
    if lr_scales[0] > 0.0:
        groups.append({"params": [p for p in enc_p if p.requires_grad], "lr": cfg.lr_encoder * lr_scales[0]})
    if lr_scales[1] > 0.0:
        groups.append({"params": [p for p in dec_p if p.requires_grad], "lr": cfg.lr_decoder * lr_scales[1]})
    if lr_scales[2] > 0.0:
        groups.append({"params": [p for p in cls_p if p.requires_grad], "lr": cfg.lr_cls_head * lr_scales[2]})
 
    optimizer = AdamW(groups, weight_decay=cfg.weight_decay)
 
    # Compute steps
    effective_steps = math.ceil(n_batches / cfg.grad_accum_steps)
    total_steps     = effective_steps * n_epochs
    warmup_steps    = max(1, int(cfg.warmup_ratio * total_steps))
 
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    print(f"Optimizer stage={stage} | total_steps={total_steps} "
          f"warmup={warmup_steps} | Active LR Groups={[g['lr'] for g in groups]}")
    return optimizer, scheduler

In [ ]:
# 15) Training loop
def train_one_epoch(
    model, loader, optimizer, scheduler, scaler, beta_cls: float
) -> Tuple[float, float, float]:
    model.train()
    run_total = run_gen = run_cls = 0.0
    optimizer.zero_grad()
    n_steps = len(loader)
 
    for step, batch in enumerate(tqdm(loader, desc="  Train", leave=False)):
        n_ids  = batch["neutral_ids"].to(cfg.device, non_blocking=True)
        n_mask = batch["neutral_mask"].to(cfg.device, non_blocking=True)
        l_ids  = batch["labeled_ids"].to(cfg.device, non_blocking=True)
        l_mask = batch["labeled_mask"].to(cfg.device, non_blocking=True)
        lbl    = batch["labels"].to(cfg.device, non_blocking=True)
        elbl   = batch["error_labels"].to(cfg.device, non_blocking=True)
 
        with _autocast():
            out      = model(n_ids, n_mask, l_ids, l_mask, labels=lbl)
            cls_loss = focal_loss_fn(out["cls_logits"], elbl)
            gen_loss = out["gen_loss"]
 
            if gen_loss is not None:
                loss = cfg.alpha_gen * gen_loss + beta_cls * cls_loss
                run_gen += gen_loss.item()
            else:
                loss = beta_cls * cls_loss
 
        # Scale loss for fp16; bf16 uses scaler as no-op
        scaled = scaler.scale(loss / cfg.grad_accum_steps)
        scaled.backward()
 
        run_total += loss.item()
        run_cls   += cls_loss.item()
 
        is_last = (step == n_steps - 1)
        if (step + 1) % cfg.grad_accum_steps == 0 or is_last:   # OPT-7
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
 
    n = len(loader)
    return run_total / n, run_gen / n, run_cls / n

In [ ]:
# 16) Validation loop
@torch.no_grad()
def validate(
    model, loader, beta_cls: float, num_beams: int = 1
) -> Dict[str, Any]:
    model.eval()
    val_total = val_gen = val_cls = 0.0
    all_true, all_pred = [], []
    all_src, all_hyp, all_ref = [], [], []
 
    for batch in tqdm(loader, desc="  Val  ", leave=False):
        n_ids    = batch["neutral_ids"].to(cfg.device, non_blocking=True)
        n_mask   = batch["neutral_mask"].to(cfg.device, non_blocking=True)
        l_ids    = batch["labeled_ids"].to(cfg.device, non_blocking=True)
        l_mask   = batch["labeled_mask"].to(cfg.device, non_blocking=True)
        lbl      = batch["labels"].to(cfg.device, non_blocking=True)
        elbl     = batch["error_labels"].to(cfg.device, non_blocking=True)
        raw_ins  = list(batch["raw_input"])
        raw_tgts = list(batch["raw_target"])
 
        with _autocast():
            out      = model(n_ids, n_mask, l_ids, l_mask, labels=lbl)
            cls_loss = focal_loss_fn(out["cls_logits"], elbl)
            gen_loss = out["gen_loss"]
 
        val_total += (cfg.alpha_gen * gen_loss + beta_cls * cls_loss).item()
        val_gen   += gen_loss.item()
        val_cls   += cls_loss.item()
 
        # Classification from the forward pass (no generation needed)
        preds = torch.argmax(out["cls_logits"].float(), dim=-1)
        all_true.extend(elbl.cpu().tolist())
        all_pred.extend(preds.cpu().tolist())
 
        # Generation via batched two-pass inference
        _, gen_texts = model.generate_and_classify(
            neutral_ids=n_ids, neutral_mask=n_mask,
            raw_inputs=raw_ins, tokenizer=tokenizer,
            cfg=cfg, id2label=id2label, num_beams=num_beams,
        )
        all_src.extend(raw_ins)
        all_hyp.extend(gen_texts)
        all_ref.extend(raw_tgts)
 
    n    = len(loader)
    gleu = gleu_scorer.corpus_gleu(all_src, all_hyp, [[r] for r in all_ref])
    chrf = chrf_metric.compute(predictions=all_hyp, references=all_ref)["score"]
 
    return {
            "loss":         val_total / n,
            "gen_loss":     val_gen   / n,
            "cls_loss":     val_cls   / n,
            "cls_acc":      accuracy_score(all_true, all_pred),
            "cls_f1_macro": f1_score(all_true, all_pred, average="macro", zero_division=0),
            "gleu":         gleu,
            "chrf":         chrf,
            "true_cls":     all_true,
            "pred_cls":     all_pred,
        }

In [ ]:
# 17) Checkpoint helpers
def save_ckpt(model, stage: int, epoch: int, f1: float, gleu: float):
    torch.save({
        "model_state_dict": model.state_dict(),
        "label_classes":    labels_sorted,
        "best_f1":          f1,
        "best_gleu":        gleu,
        "epoch":            epoch,
        "stage":            stage,
    }, cfg.best_ckpt)
 
def harmonic_mean(a: float, b: float) -> float:
    """Combined criterion for stage-1 model selection."""
    if a + b < 1e-9:
        return 0.0
    return 2 * a * b / (a + b)

In [ ]:
# 18) Training History & Plots
history: List[Dict] = []
 
def save_plots():
    if not history:
        return
    df_h = pd.DataFrame(history)
 
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))
    fig.suptitle("AOGEC and Error Type Classification Training History", fontsize=14, fontweight="bold")
 
    for metric, ax, title in [
        ("val_cls_f1_macro", axes[0, 0], "Validation Macro-F1"),
        ("val_gleu",         axes[0, 1], "Validation GLEU"),
        ("train_gen",        axes[1, 0], "Train Generation Loss"),
        ("train_cls",        axes[1, 1], "Train Classification Loss"),
    ]:
        if metric not in df_h.columns:
            continue
        for stage in sorted(df_h["stage"].unique()):
            sub = df_h[df_h["stage"] == stage]
            ax.plot(sub.index, sub[metric], marker="o", label=f"Stage {stage}")
        ax.set_title(title); ax.set_xlabel("Epoch (global)"); ax.legend()
 
    plt.tight_layout()
    path = os.path.join(cfg.output_dir, "training_history.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f" Training plot saved → {path}")

In [ ]:
# 19) Three-Stage Curriculum Training
best_f1          = -1.0
best_gleu        = -1.0
best_combined    = -1.0
patience_counter =  0
 
# ════════════════════════════════════════════════════════════════
#  STAGE 0 — Classifier warm-up  (decoder frozen)
# ════════════════════════════════════════════════════════════════
print(f"\n{'='*65}")
print(f"STAGE 0: Classifier Warm-up  ({cfg.epochs_stage0} epochs, decoder frozen)")
print(f"{'='*65}")
 
model.freeze_decoder()
opt0, sch0 = build_optimizer_and_scheduler(
    model, len(train_loader), cfg.epochs_stage0, stage=0
)
 
for epoch in range(1, cfg.epochs_stage0 + 1):
    tr  = train_one_epoch(model, train_loader, opt0, sch0, scaler, cfg.beta_cls_stage0)
    val = validate(model, val_loader, cfg.beta_cls_stage0, num_beams=cfg.num_beams_train)
 
    history.append({"stage": 0, "epoch": epoch,
                    "train_total": tr[0], "train_gen": tr[1], "train_cls": tr[2],
                    **{f"val_{k}": v for k, v in val.items()
                       if k not in ("true_cls", "pred_cls")}})
 
    print(f"[S0 E{epoch}/{cfg.epochs_stage0}] "
          f"cls_f1={val['cls_f1_macro']:.4f} acc={val['cls_acc']:.4f} "
          f"GLEU={val['gleu']:.2f} chrF={val['chrf']:.2f} "
          f"| train_cls={tr[2]:.4f}")
 
    if val["cls_f1_macro"] > best_f1:
        best_f1 = val["cls_f1_macro"]
        save_ckpt(model, 0, epoch, best_f1, val["gleu"])
        print("  → Checkpoint saved")
 
model.unfreeze_all()
 
# ════════════════════════════════════════════════════════════════
#  STAGE 1 — Joint training
# ════════════════════════════════════════════════════════════════
print(f"\n{'='*65}")
print(f"STAGE 1: Joint Training  ({cfg.epochs_stage1} epochs)")
print(f"{'='*65}")
 
patience_counter = 0
opt1, sch1 = build_optimizer_and_scheduler(
    model, len(train_loader), cfg.epochs_stage1, stage=1
)
 
for epoch in range(1, cfg.epochs_stage1 + 1):
    tr  = train_one_epoch(model, train_loader, opt1, sch1, scaler, cfg.beta_cls_stage1)
    val = validate(model, val_loader, cfg.beta_cls_stage1, num_beams=cfg.num_beams_train)
 
    history.append({"stage": 1, "epoch": epoch,
                    "train_total": tr[0], "train_gen": tr[1], "train_cls": tr[2],
                    **{f"val_{k}": v for k, v in val.items()
                       if k not in ("true_cls", "pred_cls")}})
 
    # Harmonic mean of F1 and normalised GLEU as combined criterion
    combined = harmonic_mean(val["cls_f1_macro"], val["gleu"] / 100.0)
 
    print(f"\n[S1 E{epoch}/{cfg.epochs_stage1}] "
          f"cls_f1={val['cls_f1_macro']:.4f} acc={val['cls_acc']:.4f} "
          f"GLEU={val['gleu']:.2f} chrF={val['chrf']:.2f} "
          f"combined={combined:.4f}"
          f"\n  train_gen={tr[1]:.4f} train_cls={tr[2]:.4f}")
 
    if combined > best_combined:
        best_combined = combined
        best_f1       = val["cls_f1_macro"]
        best_gleu     = val["gleu"]
        patience_counter = 0
        save_ckpt(model, 1, epoch, best_f1, best_gleu)
        print(f"  → Checkpoint saved  (combined={best_combined:.4f}  "
              f"f1={best_f1:.4f}  GLEU={best_gleu:.2f})")
    else:
        patience_counter += 1
        print(f"  → No improvement ({patience_counter}/{cfg.patience})")
        if patience_counter >= cfg.patience:
            print("  → Early stopping."); break
 
# ════════════════════════════════════════════════════════════════
#  STAGE 2 — Generation refinement
# ════════════════════════════════════════════════════════════════
print(f"\n{'='*65}")
print(f"STAGE 2: Generation Refinement ({cfg.epochs_stage2} epochs)")
print(f"{'='*65}")
 
# Reload best stage-1 checkpoint
ckpt_s1 = torch.load(cfg.best_ckpt, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt_s1["model_state_dict"])
model     = model.to(cfg.device)
best_gleu = ckpt_s1.get("best_gleu", 0.0)
patience_counter = 0
 
beta_cls_s2 = cfg.beta_cls_stage1 * 0.2
opt2, sch2  = build_optimizer_and_scheduler(
    model, len(train_loader), cfg.epochs_stage2, stage=2
)
 
for epoch in range(1, cfg.epochs_stage2 + 1):
    tr  = train_one_epoch(model, train_loader, opt2, sch2, scaler, beta_cls_s2)
    val = validate(model, val_loader, beta_cls_s2, num_beams=cfg.num_beams_train)
 
    history.append({"stage": 2, "epoch": epoch,
                    "train_total": tr[0], "train_gen": tr[1], "train_cls": tr[2],
                    **{f"val_{k}": v for k, v in val.items()
                       if k not in ("true_cls", "pred_cls")}})
 
    print(f"\n[S2 E{epoch}/{cfg.epochs_stage2}] "
          f"cls_f1={val['cls_f1_macro']:.4f} "
          f"GLEU={val['gleu']:.2f} chrF={val['chrf']:.2f} "
          f"| train_gen={tr[1]:.4f}")
 
    if val["gleu"] > best_gleu:
        best_gleu = val["gleu"]
        patience_counter = 0
        save_ckpt(model, 2, epoch, val["cls_f1_macro"], best_gleu)
        print(f"  → Checkpoint saved  (GLEU={best_gleu:.2f})")
    else:
        patience_counter += 1
        print(f"  → No GLEU improvement ({patience_counter}/{cfg.patience})")
        if patience_counter >= cfg.patience:
            print("  → Early stopping."); break
 
# Save history 
history_path = os.path.join(cfg.output_dir, "training_history.json")
_history_serialisable = [
    {k: (v if not isinstance(v, list) else None) for k, v in h.items()}
    for h in history
]
with open(history_path, "w") as f:
    json.dump(_history_serialisable, f, indent=2)
 
save_plots()   # OPT-10
print(f"\nTraining complete. History → {history_path}")


STAGE 0: Classifier Warm-up  (4 epochs, decoder frozen)
  Optimizer stage=0 | total_steps=1584 warmup=95 | Active LR Groups=[0.0, 0.0]


  Train:   0%|          | 0/1581 [00:00<?, ?it/s]

  Val  :   0%|          | 0/200 [00:00<?, ?it/s]

[S0 E1/4] cls_f1=0.4858 acc=0.5694 GLEU=13.10 chrF=84.75 | train_cls=1.1613
  → Checkpoint saved


  Train:   0%|          | 0/1581 [00:00<?, ?it/s]

  Val  :   0%|          | 0/200 [00:00<?, ?it/s]

... (42 outputs left out)


[S2 E2/3] cls_f1=0.9101 GLEU=70.17 chrF=92.06 | train_gen=0.1164
  → Checkpoint saved  (GLEU=70.17)


  Train:   0%|          | 0/1581 [00:00<?, ?it/s]

  Val  :   0%|          | 0/200 [00:00<?, ?it/s]


[S2 E3/3] cls_f1=0.9102 GLEU=70.41 chrF=92.07 | train_gen=0.1108
  → Checkpoint saved  (GLEU=70.41)
  📊 Training plot saved → /kaggle/working/aogec_v4_output/training_history.png

Training complete. History → /kaggle/working/aogec_v4_output/training_history.json


In [ ]:
# 20) Final Test Evaluation 
ckpt = torch.load(cfg.best_ckpt, map_location="cpu", weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model         = model.to(cfg.device).eval()
label_classes = ckpt["label_classes"]
label2id      = {l: i for i, l in enumerate(label_classes)}
id2label      = {i: l for l, i in label2id.items()}
 
print(f"\nBest model: Stage {ckpt['stage']} | "
      f"F1={ckpt['best_f1']:.4f} | GLEU={ckpt.get('best_gleu', 0):.2f}")
 
all_true_cls, all_pred_cls = [], []
all_src, all_ref, all_hyp  = [], [], []
rows = []
 
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        n_ids      = batch["neutral_ids"].to(cfg.device, non_blocking=True)
        n_mask     = batch["neutral_mask"].to(cfg.device, non_blocking=True)
        raw_inputs = list(batch["raw_input"])
        raw_tgts   = [post_process(t) for t in batch["raw_target"]]
        true_ids   = batch["error_labels"].cpu().tolist()
 
        with _autocast():
            cls_preds, gen_texts = model.generate_and_classify(
                neutral_ids=n_ids, neutral_mask=n_mask,
                raw_inputs=raw_inputs, tokenizer=tokenizer,
                cfg=cfg, id2label=id2label,
                num_beams=cfg.num_beams_test,
            )
 
        for i in range(len(raw_inputs)):
            all_true_cls.append(true_ids[i])
            all_pred_cls.append(cls_preds[i].item())
            all_src.append(raw_inputs[i])
            all_ref.append(raw_tgts[i])
            all_hyp.append(gen_texts[i])
            rows.append([
                raw_inputs[i],
                batch["raw_target"][i],
                id2label[true_ids[i]],
                id2label[cls_preds[i].item()],
                gen_texts[i],
            ])
 
# ── Metrics ───────────────────────────────────────────────────────────
test_gleu = gleu_scorer.corpus_gleu(all_src, all_hyp, [[r] for r in all_ref])
test_chrf = chrf_metric.compute(predictions=all_hyp, references=all_ref)["score"]

print("\n" + "─" * 50)
print("GENERATION PERFORMANCE")
print(f"  GLEU  : {test_gleu:.2f}")
print(f"  chrF  : {test_chrf:.2f}")

print("\nCLASSIFICATION PERFORMANCE")
print(classification_report(
    all_true_cls, all_pred_cls,
    target_names=label_classes, digits=4, zero_division=0,
))
 
# ── Confusion matrix ──────────────────────────────────────────────────
cm = confusion_matrix(all_true_cls, all_pred_cls)
n  = len(label_classes)
fig, ax = plt.subplots(figsize=(max(8, n), max(6, n - 1)))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=label_classes, yticklabels=label_classes, ax=ax,
)
ax.set_title("Afaan Oromoo Error Classification — Confusion Matrix")
ax.set_ylabel("Actual"); ax.set_xlabel("Predicted")
plt.tight_layout()
cm_path = os.path.join(cfg.output_dir, "test_confusion_matrix.png")
plt.savefig(cm_path, dpi=300)
plt.show()
# plt.close()
print(f"Confusion matrix saved → {cm_path}")
 
# ── Per-class GLEU ────────────────────────────────────────────────────
rows_df = pd.DataFrame(rows, columns=["input","target","true_error","pred_error","prediction"])
print("\nPER-CLASS GLEU")
for cls_name in label_classes:
    sub = rows_df[rows_df["true_error"] == cls_name]
    if not len(sub):
        continue
    g = gleu_scorer.corpus_gleu(
        sub["input"].tolist(),
        sub["prediction"].tolist(),
        [[post_process(t)] for t in sub["target"].tolist()],
    )
    acc = (sub["true_error"] == sub["pred_error"]).mean()
    print(f"  {cls_name:<42} GLEU={g:5.2f}  cls_acc={acc:.3f}  (n={len(sub)})")
 
out_path = os.path.join(cfg.output_dir, "final_test_predictions.csv")
rows_df.to_csv(out_path, index=False)
print(f"\nPredictions CSV → {out_path}")

In [ ]:
# 21) Single-sentence inference
@torch.no_grad()
def predict_single(text: str, num_beams: int = 4) -> Tuple[str, str]:
    """
    Predict error type and generate correction for one sentence.
    Uses the same two-pass (classify then generate) logic as batch inference.
    """
    model.eval()
    enc = tokenizer(
        build_neutral_prompt(text), return_tensors="pt",
        truncation=True, max_length=cfg.max_input_len, padding="max_length",
    ).to(cfg.device)
 
    with _autocast():
        cls_preds, gen_texts = model.generate_and_classify(
            neutral_ids=enc["input_ids"], neutral_mask=enc["attention_mask"],
            raw_inputs=[text], tokenizer=tokenizer,
            cfg=cfg, id2label=id2label, num_beams=num_beams,
        )
    return id2label.get(cls_preds[0].item(), "unknown"), gen_texts[0]
 
 
# ── Demo ──────────────────────────────────────────────────────────────
demo_samples = [
    "Abubakir fi Galgaloon bara kana barataa cimoo jedhamuu badhaafame turre .",
    "Inistiitiyuutiin Siinqee dubartootaa guyyaa idil addunyaa kabaje .",
    "Waan hundumaa darbuuf barbaachisa obsa .",
    "Ijaarani jiru isaan mana haaraa.",
    "Isheen kolfanii hojii ishee xumurte.",
]
 
print("\n" + "─" * 65)
print("DEMO — Afaan Oromoo GEC ")
print("─" * 65)
for text in demo_samples:
    pred_error, correction = predict_single(text)
    print(f"Input      : {text}")
    print(f"Error type : {pred_error}")
    print(f"Correction : {correction}")
    print("─" * 65)


─────────────────────────────────────────────────────────────────
DEMO — Afaan Oromoo GEC v4
─────────────────────────────────────────────────────────────────
Input      : Abubakir fi Galgaloon bara kana barataa cimoo jedhamuu badhaafame turre .
Error type : subject-verb-agreement-error
Correction : Abubakir fi Galgaloon bara kana barattoota cimoo jedhamuun badhaafamanii turan.
─────────────────────────────────────────────────────────────────
Input      : Inistiitiyuutiin Siinqee dubartootaa guyyaa idil addunyaa kabaje .
Error type : subject-verb-agreement-error
Correction : Inistiitiyuutiin Siinqee dubartootaa guyyaa idil addunyaan kabaje.
─────────────────────────────────────────────────────────────────
Input      : Waan hundumaa darbuuf barbaachisa obsa .
Error type : subject-verb-agreement-error
Correction : waan hundumaa darbuuf barbaachisa obse.
─────────────────────────────────────────────────────────────────
Input      : Ijaarani jiru isaan mana haaraa.
Error type : word-order